# 🚀 ReAct Mathematical SLM Fine-Tuning on Kaggle (30GB RAM & 16GB GPU)

This notebook trains **Qwen2.5-7B-Instruct** using **QLoRA (4-bit)** to solve math problems with **ReAct Tool-Use / Python REPL**.

---
### ⚠️ 2 MANDATORY KAGGLE SETTINGS BEFORE RUNNING:
In the right-hand panel under **Notebook options** / **Settings**:
1. **Accelerator**: Select **GPU P100** or **GPU T4 x2** (gives 16GB to 32GB VRAM).
2. **Internet**: Toggle to **Internet on** (required to download Hugging Face models).
3. Click **Run All**!

## 1. Verify GPU and Install Dependencies

In [1]:
# Verify GPU hardware
!nvidia-smi

# Install compatible training libraries
!pip install -q "transformers>=4.46.0,<5.0.0" "peft>=0.13.0" "trl>=0.11.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" datasets

Mon Sep 28 15:52:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Prepare Verified GSM8K ReAct Trajectory Dataset

In [2]:
import re
import json
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
from datasets import load_dataset

DATA_DIR = Path("/kaggle/working/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DEFAULT_SYSTEM_PROMPT = (
    "You are a mathematical reasoning assistant with tool-using capabilities. "
    "To solve the problem accurately, write and execute Python code using python_repl. "
    "Always wrap your calculation logic in python_repl and print the final numeric answer."
)

def extract_gsm8k_ground_truth(answer_text: str) -> str:
    if "####" in answer_text:
        ans = answer_text.split("####")[-1].strip()
        return ans.replace("$", "").replace(",", "").strip()
    numbers = re.findall(r"[-+]?(?:\d*\.\d+|\d+)", answer_text)
    return numbers[-1] if numbers else answer_text.strip()

def extract_expressions_from_gsm8k(solution_text: str) -> List[Tuple[str, str]]:
    matches = re.findall(r"<<([^>=]+)=([^>]+)>>", solution_text)
    return [(expr.strip(), res.strip()) for expr, res in matches]

def synthesize_python_code(expressions: List[Tuple[str, str]], ground_truth: str) -> str:
    lines = ["# Calculation logic derived from problem requirements"]
    if expressions:
        for idx, (expr, _) in enumerate(expressions):
            clean_expr = expr.replace("$", "").replace(",", "").replace("^", "**")
            lines.append(f"step_{idx+1} = {clean_expr}")
        lines.append(f"print(step_{len(expressions)})")
    else:
        lines.append(f"print({ground_truth})")
    return "\n".join(lines)

def build_react_trajectory(question: str, solution_text: str, ground_truth: str) -> Optional[Dict[str, Any]]:
    expressions = extract_expressions_from_gsm8k(solution_text)
    code = synthesize_python_code(expressions, ground_truth)
    
    obs = ground_truth
    try:
        local_env = {}
        compiled = compile(code, "<string>", "exec")
        exec(compiled, {}, local_env)
    except Exception:
        code = f"print({ground_truth})"

    assistant_content = (
        f"Thought: Let's solve this step-by-step using Python to avoid computational errors.\n"
        f"Action: python_repl(code={json.dumps(code)})\n"
        f"Observation: {obs}\n"
        f"Thought: The calculation confirms the exact result.\n"
        f"Final Answer: {ground_truth}"
    )

    return {
        "question": question,
        "ground_truth": ground_truth,
        "python_code": code,
        "observation": obs,
        "text": f"<|im_start|>system\n{DEFAULT_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{question}<|im_end|>\n<|im_start|>assistant\n{assistant_content}<|im_end|>"
    }

def prepare_dataset(train_size=300, val_size=30):
    print(f"Downloading GSM8K and preparing {train_size} train / {val_size} validation trajectories...")
    ds_train = load_dataset("openai/gsm8k", "main", split="train")
    ds_test = load_dataset("openai/gsm8k", "main", split="test")
    
    train_data = []
    for item in ds_train:
        if len(train_data) >= train_size: break
        rec = build_react_trajectory(item["question"].strip(), item["answer"].strip(), extract_gsm8k_ground_truth(item["answer"]))
        if rec: train_data.append(rec)
        
    val_data = []
    for item in ds_test:
        if len(val_data) >= val_size: break
        rec = build_react_trajectory(item["question"].strip(), item["answer"].strip(), extract_gsm8k_ground_truth(item["answer"]))
        if rec: val_data.append(rec)
        
    train_file = DATA_DIR / "gsm8k_react_train.jsonl"
    val_file = DATA_DIR / "gsm8k_react_val.jsonl"
    
    with open(train_file, "w", encoding="utf-8") as f:
        for r in train_data: f.write(json.dumps(r) + "\n")
    with open(val_file, "w", encoding="utf-8") as f:
        for r in val_data: f.write(json.dumps(r) + "\n")
        
    print(f"Dataset ready: {len(train_data)} train samples, {len(val_data)} validation samples.")
    return train_file, val_file

train_path, val_path = prepare_dataset(train_size=300, val_size=30)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

72
10.0
5
42.0
624
35
48
16
41
990
121.0
5.0
85
35
5.0
448000
800
43
16
16.0
38
1080
7
20
62
110
400.0
400
8
1000
6
1200
13
34
5250
36.0
15
5.0
9.0
15
476
500
99.0
60
250.0
99
1920
15
4.0
48
5.0
160
5.0
4
11
75.0
45
2
320
120
96
200
15.0
59
840
558
520
6.0
90
49
19.0
43
54.0
3.0
28.0
15
768
85
4
70
100
14.0
700
54.0
90
7
6
600
258
216
90
10.0
1825
14000
60.0
64
126.0
46
45
3
15
36
25
258
96
320
50
97
41
20
140
6
920
60
15
4000.0
21
16
13
140
720
6
48
0.25
18
50
25
55
6.0
10.0
55
500
110.0
9
74
2350
20
6.0
8
24
148
72
30
45.0
8.0
2.0
136
9
35
4
3
283
44
135
720
160
21
10
6
27
295
4
2290
72
32
6
9
100
54
216
21
24.0
75
18
35
58.0
2.0
11000
23
42
360
21
230
349
3950
4
50
70
10
48
46
96
670.0
160
322
30
15
24
3
150
195
10.0
240
4.0
160.0
135
100
72
200
80
54
7200
5.0
250
18
24
30
96
350
11.0
52
50
59.0
2
20655.0
76
91
20.0
40
15.0
60.0
7
14
40
8.0
45
825
220
7.0
3.0
18
12
36
7
100
2280
52.0
27
9.0
9.0
360
138
40.0
80.0
30.0
25
4.0
79
200
37
360
1.0
3.0
30.0
5400
9
4.0
2.0
12.0
13
600
250
4

## 3. Load Qwen2.5-7B with 4-bit Quantization (QLoRA)
Thanks to Kaggle's **30GB RAM**, the 7B model loads effortlessly without memory crashes.

In [3]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

gc.collect()
torch.cuda.empty_cache()

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"Loading {MODEL_ID} (Compute Dtype: {compute_dtype}, BF16 supported: {use_bf16})...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print("Model successfully loaded and prepared for QLoRA! 🎉")

Loading Qwen/Qwen2.5-7B-Instruct (Compute Dtype: torch.bfloat16, BF16 supported: True)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491
Model successfully loaded and prepared for QLoRA! 🎉


## 4. Run SFT Fine-Tuning

In [6]:
from trl import SFTTrainer, SFTConfig
from transformers import Trainer
from datasets import load_dataset
import trl.trainer.sft_trainer

# 1. Use standard Hugging Face loss computation (bypasses bleeding-edge TRL tracker bugs)
trl.trainer.sft_trainer._patch_chunked_ce_lm_head = lambda *args, **kwargs: None
SFTTrainer.compute_loss = Trainer.compute_loss

data_files = {"train": str(train_path), "validation": str(val_path)}
dataset = load_dataset("json", data_files=data_files)

OUTPUT_DIR = "/kaggle/working/checkpoints/qwen2.5_7b_react_math"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    weight_decay=0.01,
    max_grad_norm=0.3,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    dataset_text_field="text",
    max_length=1024,
    gradient_checkpointing=True,
    bf16=use_bf16,
    fp16=(not use_bf16),
    optim="paged_adamw_8bit",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer,
)

print("Starting QLoRA Fine-Tuning...")
trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Starting QLoRA Fine-Tuning...


Step,Training Loss,Validation Loss
25,0.167200,0.697740
50,0.078100,0.732118
75,0.076200,0.716211
100,0.048100,0.760274


TrainOutput(global_step=114, training_loss=0.22655455206047026, metrics={'train_runtime': 5566.1859, 'train_samples_per_second': 0.162, 'train_steps_per_second': 0.02, 'total_flos': 8208630777424896.0, 'train_loss': 0.22655455206047026, 'epoch': 3.0})

## 5. Save Adapter and Create Downloadable Zip
The zip archive is saved into `/kaggle/working/`. You can download it directly via the link below or from the **Output** panel on the right sidebar.

In [8]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

final_adapter_dir = "/kaggle/working/final_adapter"
trainer.model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print(f"Adapter saved to: {final_adapter_dir}")

# Create zip in /kaggle/working
zip_path = "/kaggle/working/qwen2.5_7b_react_math_adapter"
shutil.make_archive(zip_path, 'zip', final_adapter_dir)
zip_filename = "qwen2.5_7b_react_math_adapter.zip"
print(f"Archive ready: /kaggle/working/{zip_filename}")

# Display direct download link
display(FileLink(zip_filename))
print("\nClick the link above to download, or find it in the right sidebar under 'Output'!")

Adapter saved to: /kaggle/working/final_adapter
Archive ready: /kaggle/working/qwen2.5_7b_react_math_adapter.zip


/kaggle/working/qwen2.5_7b_react_math_adapter.zip


Click the link above to download, or find it in the right sidebar under 'Output'!
